# Fase 4 — Eventos intensos e extremos

Objetivo: identificar quais condições dos preditores ERA5, incluindo diagnósticos derivados, diferenciam eventos de radar mais intensos. São usadas definições fixas e quantílicas. Os limiares fixos permanecem em **unidades numéricas da legenda do radar** até confirmação da unidade física.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

def resolve_output(rel):
    candidates = [Path(rel), Path("..") / rel, Path.cwd() / rel, Path.cwd().parent / rel]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return Path(rel).resolve()

OUT = resolve_output("analysis_outputs/04_extremes")
print("Usando resultados em:", OUT)


In [ ]:
def load_parquet(name):
    path = OUT / name
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

def load_json(name):
    path = OUT / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

summary=load_json("analysis_summary.json")
catalog=load_parquet("predictor_catalog.parquet")
thresholds=load_parquet("threshold_definitions.parquet")
prevalence=load_parquet("event_prevalence.parquet")
cond=load_parquet("conditional_predictor_stats.parquet")
effects=load_parquet("effect_sizes.parquet")
deciles=load_parquet("event_rate_by_predictor_decile.parquet")
npz=OUT/"extreme_samples.npz"
data=np.load(npz,allow_pickle=False) if npz.exists() else None


## 1. Definições de eventos

In [ ]:
thresholds

Os quantis globais podem coincidir com zero por causa da grande massa de radar nulo. Por isso também são calculados quantis **somente entre valores positivos**, que descrevem melhor a intensidade condicional dos eventos.

## 2. Prevalência dos eventos

In [ ]:
prevalence[["event_id","family","threshold","event_count","event_rate"]] if not prevalence.empty else prevalence

In [ ]:
if not prevalence.empty:
    d=prevalence.sort_values("event_rate")
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(d["event_id"],d["event_rate"]); ax.set_xlabel("Fração de observações"); plt.show()

## 3. Diferença padronizada entre evento e não-evento

In [ ]:
if not effects.empty:
    # Prioriza um limiar físico-numérico intermediário; cai para o primeiro disponível.
    event_id="fixed_ge_30" if "fixed_ge_30" in set(effects.event_id) else effects.event_id.iloc[0]
    d=effects[effects.event_id==event_id].copy(); d["abs_smd"]=d["standardized_mean_difference"].abs(); d=d.sort_values("abs_smd")
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(d["predictor"],d["standardized_mean_difference"]); ax.axvline(0,linewidth=1); ax.set_title(event_id); ax.set_xlabel("Diferença média padronizada (evento - não-evento)"); plt.show()

## 4. Matriz de efeito por limiar fixo

In [ ]:
if not effects.empty:
    d=effects[effects.family=="fixed"]
    mat=d.pivot(index="predictor",columns="event_id",values="standardized_mean_difference")
    fig,ax=plt.subplots(figsize=(11,9)); im=ax.imshow(mat.values,aspect="auto")
    ax.set_xticks(range(len(mat.columns)),mat.columns,rotation=45,ha="right")
    ax.set_yticks(range(len(mat.index)),mat.index)
    fig.colorbar(im,ax=ax,label="SMD evento - não-evento"); fig.tight_layout(); plt.show()

## 5. Estatísticas condicionais de preditores selecionados

In [ ]:
if not effects.empty and not cond.empty:
    event_id="fixed_ge_30" if "fixed_ge_30" in set(effects.event_id) else effects.event_id.iloc[0]
    top=effects[effects.event_id==event_id].assign(a=lambda d:d.standardized_mean_difference.abs()).sort_values("a",ascending=False).predictor.head(8)
    display(cond[(cond.event_id==event_id)&(cond.predictor.isin(top))][["predictor","group","n","mean","std","p25","median","p75","p95"]])

## 6. Probabilidade de extremo por decil do preditor

In [ ]:
if not effects.empty and not deciles.empty:
    event_id="fixed_ge_30" if "fixed_ge_30" in set(deciles.event_id) else deciles.event_id.iloc[0]
    top=effects[effects.event_id==event_id].assign(a=lambda d:d.standardized_mean_difference.abs()).sort_values("a",ascending=False).predictor.head(6).tolist()
    fig,axes=plt.subplots(3,2,figsize=(13,12)); axes=axes.ravel()
    for ax,name in zip(axes,top):
        d=deciles[(deciles.event_id==event_id)&(deciles.predictor==name)].sort_values("decile")
        ax.plot(d["decile"],d["event_rate"],marker="o"); ax.set_title(name); ax.set_xlabel("Decil do preditor"); ax.set_ylabel("Taxa do evento")
    fig.tight_layout(); plt.show()

## 7. Comparação entre limiares quantílicos globais e positivos

In [ ]:
if not prevalence.empty:
    q=prevalence[prevalence.family.isin(["global_quantile","positive_quantile"])][["event_id","family","threshold","event_rate"]]
    q

## 8. Síntese da Fase 4

A saída principal para orientar o CorrDiff é a combinação de: prevalência do evento, magnitude do efeito nos preditores e monotonicidade da taxa de evento por decil. Variáveis com associação global modesta podem apresentar forte separação nos eventos mais intensos.

In [ ]:
pd.DataFrame([
    {"checagem":"Limiar fixo >=30 disponível","status":"OK" if "fixed_ge_30" in set(prevalence.event_id) else "REVISAR"},
    {"checagem":"Quantis positivos disponíveis","status":"OK" if (thresholds.family=="positive_quantile").any() else "REVISAR"},
    {"checagem":"Efeitos por preditor calculados","status":"OK" if len(effects)>0 else "REVISAR"},
    {"checagem":"Curvas por decil calculadas","status":"OK" if len(deciles)>0 else "REVISAR"},
])